# Homework 3: Classification

Predicción de la variable objetivo `converted` (si el cliente se registró en la
plataforma o no) usando el dataset 2026 de *lead scoring*.

Dataset: `course_lead_scoring_2026.csv` (release 2026 fijado en el repo del curso)

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

## Carga de datos

Descargado directamente desde el repo del curso:

```bash
wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
```

In [2]:
URL = (
    "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main"
    "/cohorts/2026/data/course_lead_scoring_2026.csv"
)

df = pd.read_csv(URL)
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [3]:
df.shape

(5000, 9)

## Data preparation

Revisamos los valores faltantes y los rellenamos:

- Features categóricas → `'NA'`
- Features numéricas → `0.0`

In [4]:
categorical = ["lead_source", "industry", "employment_status", "location"]
numerical = [
    "annual_income",
    "number_of_courses_viewed",
    "interaction_count",
    "lead_score",
]

df[categorical] = df[categorical].fillna("NA")
df[numerical] = df[numerical].fillna(0.0)

df.isna().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

## Question 1

¿Cuál es la observación más frecuente (moda) de la columna `industry`?

In [5]:
df["industry"].value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [6]:
df["industry"].mode()[0]

'technology'

**Respuesta Q1: `technology`** (1173 observaciones).

## Question 2

Matriz de correlación entre las features numéricas. ¿Qué par tiene la mayor
correlación (considerando solo los pares propuestos)?

In [7]:
df[numerical].corr().round(3)

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000,0.161,0.123,0.229
number_of_courses_viewed,0.161,1.000,0.722,0.757
interaction_count,0.123,0.722,1.000,0.916
lead_score,0.229,0.757,0.916,1.000


In [8]:
pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count"),
]

for a, b in pairs:
    print(f"{a} & {b}: {df[a].corr(df[b]):.3f}")

interaction_count & lead_score: 0.916
number_of_courses_viewed & lead_score: 0.757
number_of_courses_viewed & interaction_count: 0.722
annual_income & interaction_count: 0.123


**Respuesta Q2: `interaction_count` y `lead_score`** (correlación ≈ 0.916).

## Split del dataset

60% train / 20% validation / 20% test con las llamadas exactas del enunciado.
El target `converted` no forma parte de las features.

In [9]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

In [10]:
dv = DictVectorizer(sparse=False)

train_dicts = df_train[categorical + numerical].to_dict(orient="records")
val_dicts = df_val[categorical + numerical].to_dict(orient="records")
test_dicts = df_test[categorical + numerical].to_dict(orient="records")

X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)
X_test = dv.transform(test_dicts)

y_train = df_train["converted"].values
y_val = df_val["converted"].values
y_test = df_test["converted"].values

X_train.shape, X_val.shape, X_test.shape

((3000, 28), (1000, 28), (1000, 28))

## Question 3

Mutual information entre `converted` y las variables categóricas, usando **solo
el conjunto de entrenamiento**. Redondeando a 2 decimales.

In [11]:
for c in categorical:
    score = mutual_info_score(df_train["converted"], df_train[c])
    print(f"{c:20s} {round(score, 2)}")

lead_source          0.03
industry             0.0


employment_status    0.02
location             0.0


**Respuesta Q3: `lead_source`** (0.03).

## Question 4

Regresión logística con one-hot encoding de las categóricas, entrenada sobre el
train. Accuracy en validación redondeada a 2 decimales.

In [12]:
model = LogisticRegression(solver="liblinear", C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

accuracy = accuracy_score(y_val, model.predict(X_val))
print("accuracy val:", round(accuracy, 2))

accuracy val: 0.65


**Respuesta Q4: 0.65** (0.645 redondeado).

## Question 5

Eliminación de features (*feature elimination*). Entrenamos el modelo original y
luego uno excluyendo cada feature. La diferencia es `original - sin_feature`.
La feature *menos útil* es la que produce la menor diferencia.

In [13]:
base_acc = accuracy

features_to_remove = [
    "lead_source",
    "number_of_courses_viewed",
    "interaction_count",
]

diff = {}
for feature in features_to_remove:
    cols = [c for c in categorical + numerical if c != feature]

    dv_f = DictVectorizer(sparse=False)
    X_tr_f = dv_f.fit_transform(df_train[cols].to_dict(orient="records"))
    X_va_f = dv_f.transform(df_val[cols].to_dict(orient="records"))

    m = LogisticRegression(
        solver="liblinear", C=1.0, max_iter=1000, random_state=42
    )
    m.fit(X_tr_f, y_train)
    acc_f = accuracy_score(y_val, m.predict(X_va_f))

    d = base_acc - acc_f
    diff[feature] = d
    print(f"{feature:28s} sin_feature={acc_f:.4f}  diff={d:.4f}")

least_useful = min(diff, key=diff.get)
print("\nmenos útil:", least_useful, "->", round(diff[least_useful], 4))

lead_source                  sin_feature=0.6420  diff=0.0030
number_of_courses_viewed     sin_feature=0.6430  diff=0.0020


interaction_count            sin_feature=0.6010  diff=0.0440

menos útil: number_of_courses_viewed -> 0.002


**Respuesta Q5: `number_of_courses_viewed`** (diferencia = 0.002).

## Question 6

Regresión logística regularizada con distintos valores de `C`, usando todas las
features. Accuracy en validación redondeada a 3 decimales.

In [14]:
results = []
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    m = LogisticRegression(
        solver="liblinear", C=C, max_iter=1000, random_state=42
    )
    m.fit(X_train, y_train)
    acc = round(accuracy_score(y_val, m.predict(X_val)), 3)
    results.append((C, acc))
    print(f"C = {C:<9} accuracy = {acc}")

best_C = max(results, key=lambda t: (t[1], -t[0]))
print("\nmejor C:", best_C[0], "->", best_C[1])

C = 1e-06     accuracy = 0.598
C = 1e-05     accuracy = 0.598
C = 0.0001    accuracy = 0.613
C = 0.001     accuracy = 0.645

mejor C: 0.001 -> 0.645


**Respuesta Q6: `C = 0.001`** (accuracy = 0.645).

## Resumen de respuestas

In [15]:
summary = pd.DataFrame(
    [
        ("Q1", "moda de industry", "technology"),
        ("Q2", "mayor correlación", "interaction_count & lead_score"),
        ("Q3", "mayor mutual information", "lead_source"),
        ("Q4", "accuracy logística", 0.65),
        ("Q5", "feature menos útil", "number_of_courses_viewed"),
        ("Q6", "mejor C", 0.001),
    ],
    columns=["pregunta", "concepto", "respuesta"],
).set_index("pregunta")

summary

,concepto,respuesta
pregunta,,
Q1,moda de industry,technology
Q2,mayor correlación,interaction_count & lead_score
Q3,mayor mutual information,lead_source
Q4,accuracy logística,0.65
Q5,feature menos útil,number_of_courses_viewed
Q6,mejor C,0.001
